<a href="https://colab.research.google.com/github/andrellyrichelly-hub/ATIVIDADE-PROFESSOR-CLAUDOMIRO_-ALUNA-DE-IA-ANDRELLY-UFPA/blob/main/ATIVI_2_5_2_MEMORIA_ANDRELLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Atividade 2.5 - Memória 2: Chip DRAM 4096K x 1 - 4 Mbits
# Andrelly Richelly - Arquitetura com Claudomiro

class ChipDRAM_4096Kx1:
    """
    Simula um chip DRAM 4.096K x 1 bit = 4 Mbits
    Endereçamento multiplexado: 11 linhas A0-A10 para LINHA + COLUNA
    Total de bits de endereço: 11 + 11 = 22 bits = 4.194.304 posições
    """
    def __init__(self):
        self.linhas = 2**11      # 2048 linhas - A0 a A10
        self.colunas = 2**11     # 2048 colunas - A0 a A10
        self.total_bits = self.linhas * self.colunas  # 4.194.304 bits = 4 Mbits
        # Usar dicionário pra economizar RAM no Colab, só guarda bit=1
        self.mem = {}

    def decodificar_endereco(self, endereco_22bits):
        """
        Endereço de 22 bits: [Linha 10-0 | Coluna 10-0]
        RAS captura os 11 bits altos = linha
        CAS captura os 11 bits baixos = coluna
        """
        if not 0 <= endereco_22bits < self.total_bits:
            raise ValueError(f"Endereço {endereco_22bits} fora do limite 0-{self.total_bits-1}")

        linha = (endereco_22bits >> 11) & 0x7FF  # bits 21-11
        coluna = endereco_22bits & 0x7FF         # bits 10-0
        return linha, coluna

    def ler(self, endereco, cs=0, oe=0, ras=0, cas=0):
        """
        Leitura: CS=0, WE=1, OE=0, RAS=0->1, CAS=0->1
        Simplificado: se cs=0 e oe=0, faz leitura
        """
        if cs == 0 and oe == 0:
            linha, coluna = self.decodificar_endereco(endereco)
            dado = self.mem.get((linha, coluna), 0)  # bit 0 ou 1
            print(f"LEITURA: End={endereco:07d} | Linha={linha:04d} | Col={coluna:04d} | RAS/CAS | D={dado}")
            return dado
        else:
            print(f"LEITURA BLOQUEADA: CS={cs} OE={oe}")
            return None

    def escrever(self, endereco, dado, cs=0, we=0, ras=0, cas=0):
        """
        Escrita: CS=0, WE=0, RAS=0->1, CAS=0->1
        Dado é 1 bit: 0 ou 1
        """
        if cs == 0 and we == 0:
            linha, coluna = self.decodificar_endereco(endereco)
            dado_bit = 1 if dado else 0
            if dado_bit:
                self.mem[(linha, coluna)] = 1
            else:
                self.mem.pop((linha, coluna), None)  # economiza memória
            print(f"ESCRITA: End={endereco:07d} | Linha={linha:04d} | Col={coluna:04d} | RAS/CAS | D={dado_bit}")
        else:
            print(f"ESCRITA BLOQUEADA: CS={cs} WE={we}")

# --- TESTE AUTOMÁTICO - RODA SOZINHO NO COLAB ---
print("=== ATIVIDADE 2.5 - MEMÓRIA 2: DRAM 4096K x 1 ===\n")

dram = ChipDRAM_4096Kx1()
print(f"Chip inicializado: {dram.linhas} linhas x {dram.colunas} colunas = {dram.total_bits} bits = 4 Mbits\n")

print("Testando escrita de bits:")
dram.escrever(0, 1, cs=0, we=0)           # End 0, bit 1
dram.escrever(2048, 1, cs=0, we=0)        # Primeira posição da linha 1
dram.escrever(4194303, 1, cs=0, we=0)     # Última posição, bit 1
dram.escrever(1000000, 0, cs=0, we=0)     # Bit 0

print("\nTestando leitura:")
dram.ler(0, cs=0, oe=0)
dram.ler(2048, cs=0, oe=0)
dram.ler(4194303, cs=0, oe=0)
dram.ler(1000000, cs=0, oe=0)  # deve ser 0
dram.ler(5, cs=0, oe=0)        # nunca escrito, deve ser 0

print("\nTeste de sinais de controle:")
dram.ler(0, cs=1, oe=0)  # CS=1 bloqueia
dram.escrever(10, 1, cs=0, we=1)  # WE=1 bloqueia escrita

print("\nTeste concluído com sucesso.")

=== ATIVIDADE 2.5 - MEMÓRIA 2: DRAM 4096K x 1 ===

Chip inicializado: 2048 linhas x 2048 colunas = 4194304 bits = 4 Mbits

Testando escrita de bits:
ESCRITA: End=0000000 | Linha=0000 | Col=0000 | RAS/CAS | D=1
ESCRITA: End=0002048 | Linha=0001 | Col=0000 | RAS/CAS | D=1
ESCRITA: End=4194303 | Linha=2047 | Col=2047 | RAS/CAS | D=1
ESCRITA: End=1000000 | Linha=0488 | Col=0576 | RAS/CAS | D=0

Testando leitura:
LEITURA: End=0000000 | Linha=0000 | Col=0000 | RAS/CAS | D=1
LEITURA: End=0002048 | Linha=0001 | Col=0000 | RAS/CAS | D=1
LEITURA: End=4194303 | Linha=2047 | Col=2047 | RAS/CAS | D=1
LEITURA: End=1000000 | Linha=0488 | Col=0576 | RAS/CAS | D=0
LEITURA: End=0000005 | Linha=0000 | Col=0005 | RAS/CAS | D=0

Teste de sinais de controle:
LEITURA BLOQUEADA: CS=1 OE=0
ESCRITA BLOQUEADA: CS=0 WE=1

Teste concluído com sucesso.
